# Solutions · Chapter 04-07 · Pipelines and cross-validation

Worked answers for `notebooks/04_workflow/04-07_pipelines_cv.ipynb`.

E9 and E20 both contradict something established earlier in the course, and both are right to.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import (GridSearchCV, KFold, RandomizedSearchCV,
                                     cross_val_score, cross_validate)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

raw = fetch_california_housing(as_frame=True)
housing = raw.frame.copy()
rng = np.random.default_rng(3)
housing["region"] = pd.cut(housing.Latitude, bins=[32, 34, 36, 38, 40, 43],
                           labels=["far south", "south", "central", "north", "far north"]).astype(str)
housing.loc[rng.random(len(housing)) < 0.10, "MedInc"] = np.nan
housing.loc[rng.random(len(housing)) < 0.08, "HouseAge"] = np.nan

NUMERIC = [c for c in raw.frame.columns if c != "MedHouseVal"]
CATEGORICAL = ["region"]
sample = housing.sample(4000, random_state=0)
features, target = sample[NUMERIC + CATEGORICAL], sample.MedHouseVal
folds = KFold(5, shuffle=True, random_state=0)
inner_folds = KFold(4, shuffle=True, random_state=1)

preprocess = ColumnTransformer([
    ("numeric", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), NUMERIC),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)])
model = Pipeline([("prepare", preprocess), ("estimate", Ridge())])


def error_of(estimator):
    return -cross_val_score(estimator, features, target, cv=folds,
                            scoring="neg_mean_absolute_error").mean()


print("%d districts; ridge pipeline MAE %.4f" % (len(sample), error_of(model)))

## Quick understanding

### E1

Given a **`Pipeline`**, `cross_val_score` refits **every step** on each fold's training rows - imputer,
scaler, encoder, selector and model. Nothing in the pipeline has seen the fold's test rows.

Given a **bare model with preprocessing already applied**, it refits **only the model**. The
preprocessing was fitted once, outside, on everything - and the cross-validator has no way to know it
exists, let alone to refit it.

That asymmetry is the whole safety property: **a cross-validator can only protect what you put inside
it.**

### E2

It addresses the `strategy` parameter of the step named `impute`, which lives inside the pipeline named
`numeric`, which is a branch of the `ColumnTransformer` named `prepare`. Each `__` descends one level.

### E3

**It measures** the cross-validated score of the best candidate, on the folds that selected it.

**It does not measure** the performance you should expect from that model on new data, because the
maximum of several noisy numbers is biased upward by however much the selection exploited noise.

**In this chapter the 0.024 gap between `best_score_` and the nested score was not that bias at all.**
The selection premium measured exactly +0.0000; the whole gap came from the inner loop being 4-fold
(training on 3/4 of the rows) while the outer was 5-fold (4/5). Two things had changed, and only one of
them was the one under investigation.

## Hand calculation

### E4

`3 x 2 x 4` = **24 combinations**.

- **Plain search:** `24 x 5` = 120 fits during the search, plus one refit of the winner on all the data =
  **121 fits**.
- **Nested:** each of 5 outer folds runs a complete search with a 4-fold inner loop: `24 x 4 + 1` = 97
  fits. Five of those = **485 fits**.

### E5

At 0.8 seconds per fit: plain `121 x 0.8` = **96.8 seconds**; nested `485 x 0.8` = **388 seconds**.

**Ratio 4.0**, against the roughly 5 measured in the chapter. The measured ratio is higher because the
count ignores fixed costs that the nested version pays five times - splitting, transforming, and the
overhead of constructing 5 `GridSearchCV` objects - while the fit itself is fast for a ridge model. On a
slower estimator the measured ratio would converge towards the theoretical 4.

The general formula is worth remembering: **nesting multiplies the cost by roughly the number of outer
folds.**

### E6

`1.7 x 0.05` = **0.085** of expected optimism, against a **measured selection premium of 0.0000**.

The estimate is not merely too big, it is wrong by everything, and the reason is the assumption it makes:
that the fourteen scores are **fourteen independent draws**. They are nothing of the kind. Ridge with
alpha 0.001 and alpha 0.01 produce nearly identical models on this data, and `mean` versus `median`
imputation changes almost nothing - so the fourteen scores move together almost perfectly.

**The premium counts effective independent candidates, and fourteen highly correlated settings are close
to one.** That is why 04-03's experiment - random four-column subsets drawn from a table with twenty
noise columns - produced a large premium from forty candidates while fourteen alphas here produce none.
The candidates there were genuinely different and mostly worthless.

### E7

`8 + 5` = **13 columns**. With `drop="first"` the encoder emits four region columns instead of five, so
**12**.

**Why you might want that for a linear model:** the five one-hot columns always sum to 1, which is exactly
the intercept column, so the design matrix is rank-deficient - there are infinitely many coefficient sets
giving identical predictions. Dropping one level removes the redundancy and makes the coefficients
interpretable as differences from the dropped level.

For a **regularised** linear model it matters less (the penalty picks one solution) and for trees it does
not matter at all. It matters most when you intend to *read* the coefficients.

In [ ]:
combinations = 3 * 2 * 4
print("E4  plain : %d combos x 5 folds + 1 refit = %d fits" % (combinations, combinations * 5 + 1))
print("    nested: 5 outer x (%d x 4 + 1)         = %d fits" % (combinations, 5 * (combinations * 4 + 1)))
print("E5  at 0.8s per fit: %.1fs and %.1fs, ratio %.1f"
      % (121 * 0.8, 485 * 0.8, 485 / 121))
print("E6  1.7 x 0.05 = %.3f expected, against a measured premium of 0.0000" % (1.7 * 0.05))
print("E7  8 + 5 = 13 columns; with drop='first', 8 + 4 = 12")

## Coding

### E8 - the pipeline handles raw, messy input

In [ ]:
messy = features.iloc[:5].copy()
messy.iloc[0, messy.columns.get_loc("MedInc")] = np.nan
print("input has %d missing values and a text column of dtype %s"
      % (int(messy.isna().sum().sum()), messy.region.dtype))

fitted = model.fit(features, target)
print("predictions from raw input, no manual preparation:", np.round(fitted.predict(messy), 3))

### E9 - predicting the log of the target

In [ ]:
logged = Pipeline([("prepare", preprocess),
                   ("estimate", TransformedTargetRegressor(Ridge(), func=np.log, inverse_func=np.exp))])

print("ridge, direct            : MAE %.4f" % error_of(model))
print("ridge, on log(target)    : MAE %.4f" % error_of(logged))
print()
direct_predictions = fitted.predict(features)
logged_predictions = logged.fit(features, target).predict(features)
print("mean prediction, direct  : %.4f" % direct_predictions.mean())
print("mean prediction, logged  : %.4f" % logged_predictions.mean())
print("actual mean              : %.4f" % target.mean())

**It helps a lot here - 0.5585 against 0.6615, a 16% improvement** - which is the opposite of 04-06,
where logging the rent made things worse.

Both results are correct, and the contrast is the lesson. **A log transform is a hypothesis that effects
multiply rather than add**, and the two datasets differ in exactly that respect:

- 04-06's rents were *built* additively - area plus age plus a lift - so imposing a multiplicative form
  fought the data.
- California house values are a real, strongly right-skewed price variable where income effects are
  proportional. The hypothesis fits.

**You cannot tell which you have by looking at the skew.** Both targets were right-skewed. The test is
whether the transform helps, and it costs one line to find out.

04-06's second finding still applies: the back-transformed mean prediction sits **below** the true mean,
because `exp` of a mean of logs is a geometric mean. It is a smaller effect here than the accuracy gain,
so the transform is still worth it - but if you were summing these predictions into a total valuation,
you would want the smearing correction.

### E10 - random search against grid search

In [ ]:
# a dense list rather than a scipy distribution, so the course keeps its dependencies small
grid_space = {"estimate__alpha": np.logspace(-3, 4, 12)}
random_space = {"estimate__alpha": np.logspace(-3, 4, 500)}

started = time.perf_counter()
grid = GridSearchCV(model, grid_space, cv=inner_folds, scoring="neg_mean_absolute_error").fit(features, target)
grid_seconds = time.perf_counter() - started

started = time.perf_counter()
random = RandomizedSearchCV(model, random_space, n_iter=12, cv=inner_folds, random_state=0,
                            scoring="neg_mean_absolute_error").fit(features, target)
random_seconds = time.perf_counter() - started

print("%-22s %10s %12s %10s" % ("", "best alpha", "best_score_", "seconds"))
print("%-22s %10.4f %12.4f %10.1f" % ("grid, 12 points", grid.best_params_["estimate__alpha"],
                                      -grid.best_score_, grid_seconds))
print("%-22s %10.4f %12.4f %10.1f" % ("random, 12 draws", random.best_params_["estimate__alpha"],
                                      -random.best_score_, random_seconds))

The two land in the same place, which is the honest outcome for a **one-dimensional** search: with a
single parameter, twelve grid points cover the range perfectly well and randomness has nothing to add.

**Random search wins when the space has several dimensions and only some of them matter.** A grid over
three parameters at four values each spends 64 fits, but tries only **four distinct values of each
parameter**. Sixty-four random draws try 64 distinct values of each. If two of the three parameters turn
out to be irrelevant - which is the normal case - the grid has wasted three quarters of its budget
re-testing the same value of the one that mattered.

So the answer is about geometry rather than luck: **grids waste budget on the axes that do not matter,
and random search does not know which axes those are but does not need to.**

### E11 and E12 - comparing pipelines on the same folds

In [ ]:
def compare_pipelines(named, X, y, cross_validator):
    rows, per_fold = [], {}
    for name, estimator in named.items():
        outcome = cross_validate(estimator, X, y, cv=cross_validator,
                                 scoring="neg_mean_absolute_error", return_train_score=True)
        per_fold[name] = -outcome["test_score"]
        rows.append({"model": name,
                     "test MAE": round(-outcome["test_score"].mean(), 4),
                     "train MAE": round(-outcome["train_score"].mean(), 4),
                     "train-test gap": round(-outcome["train_score"].mean() + outcome["test_score"].mean(), 4),
                     "fold sd": round(outcome["test_score"].std(), 4)})
    table = pd.DataFrame(rows).sort_values("test MAE")
    best = table.model.iloc[0]
    table["paired vs best"] = [round((per_fold[name] - per_fold[best]).mean(), 4) for name in table.model]
    table["paired sd"] = [round((per_fold[name] - per_fold[best]).std(), 4) for name in table.model]
    return table


candidates = {
    "ridge": Pipeline([("prepare", preprocess), ("estimate", Ridge())]),
    "random forest": Pipeline([("prepare", preprocess),
                               ("estimate", RandomForestRegressor(200, random_state=0,
                                                                  min_samples_leaf=2, n_jobs=-1))]),
    "gradient boosting": Pipeline([("prepare", preprocess),
                                   ("estimate", HistGradientBoostingRegressor(random_state=0))]),
}
comparison = compare_pipelines(candidates, features, target, folds)
print(comparison.to_string(index=False))

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12.5, 4.4))

positions = np.arange(len(comparison))
left.barh(positions + 0.19, comparison["train MAE"], height=0.36, color="#cfe3f3", label="train")
left.barh(positions - 0.19, comparison["test MAE"], height=0.36, color="#0072B2", label="test")
for position, row in enumerate(comparison.itertuples()):
    left.text(row._3 + 0.01, position + 0.19, "gap %.3f" % abs(row._4), va="center", fontsize=8.5)
left.set_yticks(positions)
left.set_yticklabels(comparison.model.str.replace(" ", "\n"), fontsize=9)
left.set_xlabel("mean absolute error")
left.set_title("Train and test, by model", fontsize=11)
left.legend(fontsize=8, loc="lower right")

right.barh(positions, comparison["train-test gap"].abs(), color="#D55E00")
for position, row in enumerate(comparison.itertuples()):
    right.text(abs(row._4) + 0.004, position, "test %.3f" % row._2, va="center", fontsize=9)
right.set_yticks(positions)
right.set_yticklabels(comparison.model.str.replace(" ", "\n"), fontsize=9)
right.set_xlabel("size of the train-test gap")
right.set_title("The biggest gap is not the worst model", fontsize=11)

plt.tight_layout()
plt.show()

**Gradient boosting wins at 0.3530, then the forest at 0.3942, then ridge at 0.6615.**

Read the paired columns, and note that they do not say what you might expect. Boosting beats the forest by
**0.0412 with a paired standard deviation of 0.0096** - more than four times its own noise, so that
ranking is solid. Boosting beats ridge by **0.3085 with a paired standard deviation of 0.1836** - a gap
seven times larger, and yet only 1.7 times its own noise.

**A bigger difference is not automatically a better-established one.** Ridge's fold-to-fold behaviour is
so erratic here (fold sd 0.1813 against boosting's 0.0129) that even a huge average gap comes with a huge
spread. The pairing removes the variation the two models share; it cannot remove variation that belongs to
one of them alone.

**The overfitting ranking does not match the performance ranking, and that is E12's point.** The random
forest has the largest train-test gap (0.222) and is *not* the worst model; ridge has the smallest gap
(0.105) and is by far the worst. A small gap can mean "generalises well" or it can mean **"underfits
equally badly on both"**, and ridge is the second.

So the gap is a diagnostic, not a score:

- **Large gap, good test score** - the forest. It memorises heavily and still generalises well. Nothing to
  fix unless you want a smaller model.
- **Small gap, poor test score** - ridge. Not an overfitting problem; a *capacity* problem. Regularising
  it further would make it worse.
- **Large gap, poor test score** - the case where reducing capacity actually helps.

Note also the fold spreads: ridge 0.1813 against boosting 0.0129. Ridge is not only worse on average, it
is wildly inconsistent across California districts - a fact the mean alone conceals, and 05-08's subject.

## Interpretation

### E13

**"0.31 MAE, five-fold cross-validated, with hyperparameters tuned by grid search"** is missing:

1. **How many candidates were tried**, and how different they were. Without it, the reader cannot judge
   whether the tuning inflated the number.
2. **Whether the 0.31 is `best_score_` or a held-out score.** If the same folds tuned and reported, it is
   the winner's score on the folds that chose it.
3. **A baseline.** 04-02's question: 0.31 against what? The target's spread, and what a constant scores.
4. **Whether the split was appropriate** - grouped, chronological, or plain - which 04-04 showed can be
   worth 0.10 of AUC on its own.
5. **The fold-to-fold spread.** A mean of 0.31 with folds ranging 0.15 to 0.55 is a different claim from
   one with folds at 0.30 to 0.32 - and this chapter's ridge had a fold sd of 0.18 on a mean of 0.66.

The first two are the ones specific to this chapter. **I would ask for the number of candidates and a
held-out score**, in that order, because the first is free to answer and often makes the second
unnecessary.

### E14

**Report the nested 0.69.**

The likely explanation is not that either number is wrong but that **the test set happened to be
easier** - which 04-03 measured as a spread of 0.21 in AUC across splits of the same data. A single
held-out set is one draw; nested cross-validation is an average over five. On a small dataset the single
number has a large standard error, and 0.07 is well inside it.

The way to settle it is to look at the nested **fold spread**. If the five outer folds range from 0.60 to
0.78, then 0.62 is an ordinary member of that distribution and there is nothing to explain. If they range
0.68 to 0.70, then 0.62 is genuinely anomalous and the test set differs from the rest in some way worth
finding - a different time period, a different region, a different collection process.

**Reporting the more pessimistic of two honest estimates is also just good practice**, since the failure
mode of over-promising is much worse than that of under-promising.

## Debugging

### E15

**`alpha` is a parameter of the estimator inside the pipeline, not of the pipeline itself.** A `Pipeline`
has no `alpha`.

The fix is to name the step: if the pipeline is `Pipeline([("prepare", ...), ("estimate", Ridge())])`,
the parameter is **`estimate__alpha`**. `model.get_params().keys()` lists every legal name, which is the
fastest way to find the right one when the nesting is deep.

### E16

**The likely step is a `KNNImputer`, or any estimator that stores training rows** - `KNeighborsRegressor`,
a kernel SVM keeping support vectors, or a large random forest.

`KNNImputer` must retain the entire training set to find neighbours at prediction time, so the pickled
pipeline contains a copy of your data. **The trade-off is accuracy against artefact size and prediction
latency**: 04-06's E9 found `KNNImputer` beating the mean by 3.39 EUR, and that gain is paid for in
megabytes and in a slower, memory-hungry prediction path.

For a batch job the trade is usually worth it. For a low-latency service it usually is not, and a simple
imputer plus a missingness indicator gets part of the benefit for none of the cost.

## Exam and interview reasoning

### E17

> "Three reasons. First, correctness: every fitted step - imputer, scaler, encoder, selector - gets
> refitted inside each cross-validation fold automatically, so preprocessing cannot see the held-out rows.
> Second, deployment: the fitted pipeline is one object that carries its own preparation, so serving it is
> a load and a predict on a raw DataFrame, rather than remembering to apply four transforms in the right
> order. Third, search: because the pipeline is what gets refitted, I can put preprocessing choices in the
> hyperparameter grid and tune them honestly - the imputation strategy becomes just another parameter."

**"When would you not bother?"**

> "For a quick look at whether there is any signal at all - fit something on a subset, see if it beats a
> baseline, decide whether the project is worth pursuing. That is exploratory, nobody will act on the
> number, and the pipeline can come with the first real experiment. What I would not do is skip it because
> the leakage seems small: I measured it on one dataset at 0.01 and on another at 0.25, and the code was
> identical. The size of the leak is a property of the data, not of the mistake."

## Transfer to a different situation

### E18

**The `ColumnTransformer`:**

```
ColumnTransformer([
    ("text",     TfidfVectorizer(min_df=5),                     "description"),
    ("numeric",  Pipeline([SimpleImputer(), StandardScaler()]),  ["price", "weight"]),
    ("category", OneHotEncoder(handle_unknown="ignore"),         ["brand", "category"]),
])
```

Note that the text branch takes a **single column name as a string**, not a list - vectorisers expect a
1-D sequence of documents, and this is the commonest error when adding text to a `ColumnTransformer`.

**The step most likely to leak: the text vectoriser**, and it is not close. `TfidfVectorizer` learns a
vocabulary *and* inverse document frequencies from whatever it is fitted on, and it produces thousands of
columns - so it is 04-05's wide-data regime, where a selection or a fit on all the data has the most room
to exploit noise. Fitted before the split it has seen every held-out description.

**The hyperparameter I would search first: `min_df`** on the vectoriser - the minimum number of documents
a term must appear in. It controls the width of the whole feature space, and width is what drives both
the compute cost and the overfitting risk here. Getting it roughly right matters far more than tuning the
classifier.

Worth adding: `brand` may be high-cardinality, so check rows per level (04-06's E7) before one-hot
encoding it.

## Explain it to someone non-technical

### E19

> "Think of a recipe that calls for 'two cups of flour, sifted'. If I hand you only the baking
> instructions and keep the sifting step in my head, you will get a different cake - and you will not know
> why. The model is the same: before it can make a prediction, the incoming data has to be cleaned and
> converted in exactly the way it was during training. We package the preparation and the model together
> as one thing, so whoever uses it cannot accidentally skip a step or do them in the wrong order."

(85 words.)

## Optional challenge

### E20 - does the gap grow with the grid?

In [ ]:
alphas = list(np.logspace(-3, 4, 60))
matched_inner = KFold(5, shuffle=True, random_state=1)   # same fold count as the outer loop

rows = []
for size in [2, 6, 14, 30, 60]:
    space = {"estimate__alpha": alphas[:size]}
    search = GridSearchCV(model, space, cv=matched_inner,
                          scoring="neg_mean_absolute_error").fit(features, target)
    nested = -cross_val_score(GridSearchCV(model, space, cv=matched_inner,
                                           scoring="neg_mean_absolute_error"),
                              features, target, cv=folds,
                              scoring="neg_mean_absolute_error").mean()
    chosen = Pipeline([("prepare", preprocess), ("estimate", Ridge())])
    chosen.set_params(**search.best_params_)
    flat = -cross_val_score(chosen, features, target, cv=folds,
                            scoring="neg_mean_absolute_error").mean()
    rows.append({"candidates": size,
                 "best_score_": round(-search.best_score_, 4),
                 "nested": round(nested, 4),
                 "plain CV of the chosen alpha": round(flat, 4),
                 "premium (nested - plain)": round(nested - flat, 4)})
premium_table = pd.DataFrame(rows)
print(premium_table.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.4))
ax.plot(premium_table.candidates, premium_table["premium (nested - plain)"], "o-",
        color="#0072B2", linewidth=2, markersize=8, label="measured premium here")
ax.axhline(0, color="#000000", linewidth=1)
reference = 0.12 * np.log(premium_table.candidates) / np.log(40)
ax.plot(premium_table.candidates, reference, ":", color="#D55E00", linewidth=2,
        label="04-03's premium, for scale")
ax.set_xscale("log")
ax.set_ylim(-0.02, 0.16)
ax.set_xlabel("candidates in the grid (log scale)")
ax.set_ylabel("nested minus plain CV of the chosen setting")
ax.set_title("Sixty ridge alphas exploit exactly nothing", fontsize=11.5)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**It does not grow. It is +0.0000 at every size from 2 to 30, and +0.0001 at 60.**

That flatly disagrees with 04-03's E20, where the premium climbed to 0.12 by forty candidates and stayed
there. Both measurements are correct, and reconciling them is the point of the exercise.

**The difference is what a "candidate" is.**

- 04-03's candidates were random four-column subsets drawn from a table containing twenty pure-noise
  columns. Each was a genuinely different model, most were worthless, and the best of forty was the one
  that got luckiest on 104 validation rows.
- These candidates are sixty values of a single smooth regularisation parameter, evaluated on 4,000 rows.
  They do not explore sixty directions; they trace **one curve with one minimum**. Doubling the number of
  points on that curve adds resolution, not opportunity.

> **The selection premium scales with the number of *effectively independent* candidates, and with how
> little data each is judged on.** A grid of 60 correlated settings on 4,000 rows is closer to one
> candidate than to sixty.

Two practical consequences:

- **A large grid is not automatically dangerous.** Sweeping one regularisation parameter finely is cheap
  and honest. Sweeping twenty unrelated architectural choices on a small validation set is neither.
- **Measure it rather than assuming either way.** The four-column table above costs one cell, and it
  answers "is my search inflating my number?" directly - which is more useful than any rule of thumb.

Finally, notice that `best_score_` sits at 0.6382 against the nested 0.6615 at *every* grid size, even
though the premium is zero. **That persistent 0.023 is structural**, not selection: nesting always trains
its inner models on a fraction of a fraction of the data, so nested estimates are systematically slightly
pessimistic. It is the training-size effect from the chapter, and it does not go away by matching fold
counts - which is worth knowing before you interpret a nested score as the truth.